# Late rollout consistency

Train the offline model first, then switch the consistency batch over to **model-proposed queries** at
`START_LATE`. The queries never enter a likelihood term: no relabelling, no fitting of generated rows, no
maze structure anywhere in training (`writeup/math.tex`, `TODO.md`). The real maze is used only by the
evaluation cells.

Outcomes are binned **geometrically at K = 24** (`MAZE_KW`): log-spaced arrival-time bins, so optimal play
spans ~11 usable bins instead of 2 under uniform binning, and every conditioned row carries a more specific
request. Only the labels change; the stored rollouts and the model code are untouched.

**Conditioning** (`COND`): `threshold` makes reward token k mean "bin k or faster". A row that achieved
bin b is then a valid conditioned sample for every k <= b, so the conditioned half of each batch and the
recorded-query consistency batch draw (row, satisfied threshold) pairs uniformly, and the identity's value
term is the tail sum of the categorical head. Adjacent bins share data and value mass instead of being
independent. `bin` is the original semantics.

Two proposal sources, both with the model's own reward head as the only feasibility estimate:

- `tilt` -- recorded rows, query bin drawn from the NOR reward prediction at the start tilted by
  `exp(beta * r_k)`; each row's intervals stop where the model's belief in the query bin falls below `FLOOR`
  (`consistency.make_tilted_sampler`).
- `rollout` -- the model's own reward-conditioned rollouts from recorded start cells, truncated after
  `MAX_STEPS` imagined steps (the identity needs no terminal; short rollouts stay inside the query's
  plausible window and are cheap), learned termination (the model can emit END) and NOR dynamics,
  query = the request; same belief floor (`consistency.make_rollout_sampler`).

Why late: on a 2k-step checkpoint the far-start value head is flat at ~1e-2 across bins, so the floor cannot
tell feasible from infeasible and a high query spends almost all of its intervals comparing softmax floors,
which raised the far-start tail instead of sharpening it. A longer offline phase is the hypothesis for a
value head with a real tail. Arms: `mc_all` (recorded bins throughout), `late_tilt`, `late_rollout`,
`late_both`. After `START_LATE` half of each consistency batch stays recorded-bin real rows (the term that
improves the far-start value tail); the proposals are added to it. Imagined rows get no likelihood or value
term, only the residual.


In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_PROJECT = "sillyrl-late"  #@param {type:"string"}


In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks; deterministic) and the exact test set for THIS run's outcome binning.
# Geometric binning at K=24 relabels outcomes only (log-spaced arrival-time bins: fine resolution where optimal
# play lives); the stored rollouts are untouched, but the exact test set must match the bins, so it gets its own file.
MAZE_KW = dict(binning="geometric", n_bins=24)
COND = "threshold"                    # "bin": token k = landed in bin k; "threshold": bin k or faster (nested events)
TEST_PATH = f"data/canonical/testset_geo24_{COND}.npz"
!python run.py dataset | tail -4
from maze_consistency.dataset import canonical_maze
from maze_consistency.testset import build_testset
import os
_maze = canonical_maze(**MAZE_KW)
print(_maze, "| empty bins (dead classes, skipped by the test set):", _maze.empty_bins)
if not os.path.exists(TEST_PATH):
    build_testset(_maze, path=TEST_PATH, cond=COND)


In [ ]:
# Weights & Biases. Put your key in a Colab secret named WANDB_API_KEY (or you will be prompted). One W&B run
# per training run; every key train() produces is logged as-is, so "/" groups become panel sections:
#   loss/*          tf, mc, cons (raw) and cons_weighted = LAMBDA * cons -- the magnitude question
#   diag/*          cond_gap, info_gain
#   act_kl/*, value_kl/*, cons/*     exact-test metrics per setting (score) and held-out consistency
#   enrich/*, goalward/*, goalward_bin/*   enrichment, and goalward mass when asking each bin
#   sampler/*       what the late samplers proposed: query-bin shares, rows cut by the floor, END rate
#   gnorm/*, cons_rows/*   every GRAD_EVERY steps: each term's gradient norm taken alone, the consistency
#                   term split into recorded-bin rows vs proposal rows, and the per-row loss mean/std of each
#   rollout_eval/*  EVALUATION ONLY, real maze: invalid imagined transitions, END emitted at the goal, reached
import numpy as np

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from google.colab import userdata
        wandb.login(key=userdata.get("WANDB_API_KEY"))
    except Exception:
        wandb.login()


def make_logger(run, lc, sampler, config):
    """metrics_fn for train(): forwards train parts and eval metrics to wandb, plus the samplers' readouts."""
    if not USE_WANDB:
        return None
    seen = {}
    run_id = "".join(ch if ch.isalnum() else "-" for ch in run)          # stable id: a resumed run logs into the same W&B run
    wb = wandb.init(project=WANDB_PROJECT, name=run, id=run_id, resume="allow",
                    config=dict(config, **{f"loss.{k}": v for k, v in lc.__dict__.items()}), reinit=True)

    def fn(step, m, kind):
        out = {}
        for k, v in m.items():
            if k == "step" or not np.isscalar(v):
                continue
            if kind == "train" and not k.startswith(("gnorm/", "cons_rows/")):
                out[f"loss/{k}" if k in ("tf", "mc", "td", "cons") else f"diag/{k}"] = float(v)
            else:
                out[k] = float(v)
        if kind == "train" and "cons" in m:
            out["loss/cons_weighted"] = lc.w_cons * float(m["cons"])
        if kind == "train" and sampler is not None and getattr(sampler, "late", None):
            for j, sub in enumerate(sampler.late):
                if sub.history:
                    h = sub.history[-1]
                    tag = f"sampler/{j}"
                    share = h["bins"] / max(h["bins"].sum(), 1)
                    out[f"{tag}/share_top2"] = float(share[-2:].sum())
                    out[f"{tag}/share_bin0"] = float(share[0])
                    out[f"{tag}/mean_used"] = float(h["mean_used"])
                    out[f"{tag}/cut_frac"] = float(h["n_cut"]) / max(h["bins"].sum(), 1)
                    if "ended" in h:
                        out[f"{tag}/ended"] = float(h["ended"]); out[f"{tag}/mean_len"] = float(h["mean_len"])
                    if "beta" in h:
                        out[f"{tag}/beta"] = float(h["beta"])
                if hasattr(sub, "last_rollout"):            # real-maze check of the imagined rollouts, eval only
                    ro, tau, ro_step = sub.last_rollout()
                    if ro_step != seen.get(j):
                        seen[j] = ro_step
                        out.update({f"rollout_eval/{k}": v for k, v in imagined_rollout_eval(MAZE, ro, tau).items()})
        wb.log(out, step=step)

    fn.finish = wb.finish
    return fn


## 1. Arms

Every parameter is a plain variable; edit and re-run. `late` samplers are built per run from the model.

In [ ]:
from maze_consistency.train import LossConfig
import maze_consistency.consistency as C
import numpy as np

STEPS, START_LATE = 20000, 12000     #@param
OBJECTIVE, LAMBDA, CONS_BATCH = "all_scaled", 0.14, 16
LR, WARMUP, COSINE = 3e-3, 0, True   # optimizer defaults for the rollout arms; the speed sweep below varies them
D_MODEL, N_LAYERS = 64, 2
POS_ENC = "learned"                  # or "rope": rotary attention + fixed sinusoidal absolute time (model.ModelConfig)
MODE_ENC = "free"                    # or "ordinal": reward-bin token = Fourier features of the bin's log arrival time + residual
BETA, FLOOR = 3.0, 1e-3              # small constant tilt; the floor is the model's own belief
BUFFER_N, REFRESH_EVERY, TAU_MAX = 256, 100, 0   # tau_max=0: imagined steps first, so the floor cut lands after them
MAX_STEPS = 10                       # truncated rollouts: ~20 model calls per refresh instead of ~400
REQUEST = "highest"                  # or "tilt" (uses BETA)
KEEP_RECORDED = 0.5
CKPT_EVERY = 1000                    # mid-run checkpoints to Drive; a dead session resumes from the last one
GRAD_EVERY = 100                     # per-term gradient norms (gnorm/*, cons_rows/*), a diagnostic only                  # share of each late consistency batch that stays recorded-bin real rows

cons = LossConfig(mc=True, cons=True, cons_loss=OBJECTIVE, w_cons=LAMBDA, cons_batch=CONS_BATCH)


def phased(late_names):
    def factory(model, tok, maze, data, n_train):
        late = []
        if "tilt" in late_names:
            late.append(C.make_tilted_sampler(model, tok, maze, data, n_train, beta=BETA, floor=FLOOR, truncate=True))
        if "rollout" in late_names:
            late.append(C.make_rollout_sampler(model, tok, maze, data, n_train, request=REQUEST, beta=BETA,
                                               buffer_n=BUFFER_N, refresh_every=REFRESH_EVERY, tau_max=TAU_MAX, floor=FLOOR,
                                               max_steps=MAX_STEPS))
        s = C.make_phased_sampler(tok, maze, data, n_train, late=late, start=START_LATE, keep_recorded=KEEP_RECORDED)
        s.late = late
        return s
    return factory


# A spec is (LossConfig, sampler factory or None) or (LossConfig, factory, train kwargs) -- the third element
# overrides run_sweep's defaults (lr, warmup, cosine, d_model, n_layers, batch) for that run.
SPECS = {
    "mc_all":       (cons, None),
    # "late_tilt":    (cons, phased(["tilt"])),
    # "late_rollout": (cons, phased(["rollout"])),
    # "late_both":    (cons, phased(["tilt", "rollout"])),
}
LOSSES = list(SPECS)

# Speed sweep: the base configuration (recorded bins throughout) under the untuned knobs. Shorter runs;
# the question is slope, not asymptote. Read value_kl on the far-start bins and enrichment.
from dataclasses import replace as _replace
SPEED_STEPS = 5000
SPEED_SPECS = {
    "base_1e-3":        (cons, None, dict(lr=1e-3)),
    "cos_3e-3":         (cons, None, dict(lr=3e-3, warmup=250, cosine=True)),
    "cos_3e-3_big":     (cons, None, dict(lr=3e-3, warmup=250, cosine=True, d_model=128, n_layers=4)),
    "cos_3e-3_cons32":  (_replace(cons, cons_batch=32), None, dict(lr=3e-3, warmup=250, cosine=True)),
    "cos_3e-3_lam0.5":  (_replace(cons, w_cons=0.5), None, dict(lr=3e-3, warmup=250, cosine=True)),
    "cos_3e-3_rope":    (cons, None, dict(lr=3e-3, warmup=250, cosine=True, pos_enc="rope")),
    "cos_3e-3_ordinal": (cons, None, dict(lr=3e-3, warmup=250, cosine=True, mode_enc="ordinal")),
    "cos_3e-3_rope_ord": (cons, None, dict(lr=3e-3, warmup=250, cosine=True, pos_enc="rope", mode_enc="ordinal")),
}
RUN_SPEED_SWEEP = True   #@param {type:"boolean"}


## 2. The sweep

Same loop as the other notebooks. Runs save to `RUNS_DIR/<PREFIX>/<name>_s<seed>` and are skipped if present.

In [ ]:
import os
import numpy as np
from maze_consistency.dataset import load as load_data
from maze_consistency.dp import compute_ground_truth
from maze_consistency.tokens import Tokenizer
import jax.numpy as jnp
from maze_consistency.model import ModelConfig, MazeTransformer, make_forward
from maze_consistency.testset import load_testset, stratified_rows, score
from maze_consistency.evaluate import make_enrichment_eval, imagined_rollout_eval
from maze_consistency.train import train, load_run, RUNS_DIR, N_HELDOUT

MAZE, DATA = load_data(**MAZE_KW)
LIVE = [k for k in range(MAZE.K) if k not in MAZE.empty_bins]      # usable bins
_best_at = lambda dd: int(MAZE.best_bin(np.flatnonzero(MAZE.dist == dd)[0]))
FAR_BINS = sorted({_best_at(20), _best_at(15), _best_at(10)})            # far starts' own best bins, under any binning
TOP2 = (f"bin {FAR_BINS[0]}", f"bin {FAR_BINS[-1]}")                     # panel settings: best bin at dist 20 and at dist 10
LIVE = [k for k in range(MAZE.K) if k not in MAZE.empty_bins]      # usable bins
_best_at = lambda dd: int(MAZE.best_bin(np.flatnonzero(MAZE.dist == dd)[0]))
FAR_BINS = sorted({_best_at(20), _best_at(15), _best_at(10)})            # far starts' own best bins, under any binning
TOP2 = (f"bin {FAR_BINS[0]}", f"bin {FAR_BINS[-1]}")                     # panel settings: best bin at dist 20 and at dist 10
TOK = Tokenizer(MAZE, cond=COND)
N_TRAIN = len(DATA["length"]) - N_HELDOUT
HELDOUT = np.random.default_rng(0).choice(np.arange(N_TRAIN, N_TRAIN + N_HELDOUT), 64, replace=False)
SAMPLERS = {}                                   # run name -> its phased sampler (for the readouts below)


def run_sweep(specs, seeds=(0,), steps=STEPS, batch=32, lr=LR, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=4,
              warmup=WARMUP, cosine=COSINE, pos_enc=POS_ENC, mode_enc=MODE_ENC,
              eval_every=500, eval_per_setting=50, heldout=HELDOUT, log_every=100, prefix="late", grad_every=GRAD_EVERY, ckpt_every=CKPT_EVERY,
              skip_existing=True, log=print):
    ts = load_testset(TEST_PATH)
    rows = stratified_rows(ts, eval_per_setting, seed=0)
    enrich = make_enrichment_eval(TOK, MAZE, DATA, n=128)
    defaults = dict(steps=steps, batch=batch, lr=lr, d_model=d_model, n_layers=n_layers, n_heads=n_heads,
                    warmup=warmup, cosine=cosine, pos_enc=pos_enc, mode_enc=mode_enc)

    done = {}
    for name, spec in specs.items():
        lc, factory, over = (spec + (None, {}))[:3]
        kw = dict(defaults, **(over or {}))
        cfg = ModelConfig.for_tokenizer(TOK, d_model=kw["d_model"], n_layers=kw["n_layers"], n_heads=kw["n_heads"],
                                        pos_enc=kw["pos_enc"], mode_enc=kw["mode_enc"])
        model = MazeTransformer(cfg)
        cons_eval = C.make_heldout_eval(model, TOK, MAZE, DATA, heldout)

        def eval_fn(params, fwd, cons_eval=cons_eval):
            m = score(params, fwd, TOK, ts, rows)
            m.pop("per_setting")
            m.update(cons_eval(params))
            m.update(enrich(params, fwd))
            return m

        for seed in seeds:
            run = f"{prefix}/{name}_s{seed}"
            if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
                log(f"[skip] {run} exists")
                continue
            sampler = factory(model, TOK, MAZE, DATA, N_TRAIN) if factory else None
            SAMPLERS[run] = sampler
            logger = make_logger(run, lc, sampler, dict(kw, seed=seed, start_late=START_LATE, beta=BETA, floor=FLOOR,
                                                        max_steps=MAX_STEPS, request=REQUEST,
                                                        keep_recorded=KEEP_RECORDED, cond=COND, **MAZE_KW))
            done[run] = train(name=run, seed=seed, loss=lc, eval_fn=eval_fn, eval_every=eval_every,
                              log_every=log_every, log=log, cons_sampler=sampler, metrics_fn=logger,
                              grad_every=grad_every, ckpt_every=ckpt_every, maze_kw=MAZE_KW, cond=COND, **kw)
            if logger:
                logger.finish()
    return done


# Run names carry the conditioning, so bin and threshold sweeps never collide or skip each other on Drive.
SPEED_PREFIX = f"speed_{COND}"
if RUN_SPEED_SWEEP:
    run_sweep(SPEED_SPECS, seeds=(0,), steps=SPEED_STEPS, prefix=SPEED_PREFIX)

PREFIX = f"late_{COND}"
run_sweep(SPECS, seeds=(0,), prefix=PREFIX)


## 3. Exact-test metrics and enrichment

The vertical line is `START_LATE`. Read `bin 10`, `bin 11`, `best far` for the conditioned settings; enrichment is the number that matters.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt


def load_history(prefix, which="test"):
    """{run name: [history per seed]}. which="test" -> exact-test metrics at each checkpoint;
    which="train" -> logged loss parts, including cons / cond_gap / info_gain."""
    root = os.path.join(RUNS_DIR, prefix)
    out = {}
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        p = os.path.join(root, d, "history.json")
        if os.path.exists(p):
            with open(p) as f:
                h = json.load(f)
            if not isinstance(h, dict) or which not in h:
                continue                      # pre-refactor runs stored a bare list; skip rather than crash
            out.setdefault(d.rsplit("_s", 1)[0], []).append(h[which])
    return out


def plot_curves(prefix, metrics=("act_kl", "value_kl"),
                settings=("NOR", *TOP2, "best far"), logy=True, order=None, figsize=(4.2, 3.4)):
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(len(metrics), len(settings),
                           figsize=(figsize[0] * len(settings), figsize[1] * len(metrics)), squeeze=False)
    for i, metric in enumerate(metrics):
        for j, setting in enumerate(settings):
            a, key = ax[i, j], f"{metric}/{setting}"
            for c, name in enumerate(names):
                hists = runs[name]
                steps = [m["step"] for m in hists[0]]
                ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
                a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
                if len(hists) > 1:
                    a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
            if logy:
                a.set_yscale("log")
            a.set_title(key, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=8)
    fig.tight_layout()
    return fig




def _mark(fig):
    for a in fig.axes:
        a.axvline(START_LATE, color="k", lw=.8, ls="--")

if RUN_SPEED_SWEEP:                     # speed sweep first: slope on the far-start bins and enrichment
    _L = LOSSES; LOSSES = list(SPEED_SPECS)
    plot_curves(SPEED_PREFIX); plt.show()
    LOSSES = _L
fig = plot_curves(PREFIX); _mark(fig); plt.show()


In [ ]:
from maze_consistency.evaluate import make_enrichment_eval

def enrichment_table(prefix=PREFIX, names=None, seed=0, n=256):
    """Recompute enrichment from each run's saved params, so it works for older runs too."""
    probe = make_enrichment_eval(TOK, MAZE, DATA, n=n)
    names = [n_ for n_ in (names or list(LOSSES))]
    rows = {}
    for name in names:
        try:
            params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
        except FileNotFoundError:
            continue
        rows[name] = probe(params, make_forward(MazeTransformer(mcfg), TOK))
    if not rows:
        print("no runs found for prefix", prefix); return rows
    w = max(len(k) for k in rows) + 2
    print(f"exact conditional gains {probe.ceiling:+.1f} points -- that is 100%")
    print(f"{'run':<{w}}{'ask: fail':>11}{'ask: best':>11}{'enrichment':>12}{'% of exact':>12}")
    for name, m in rows.items():
        print(f"{name:<{w}}{m['goalward/fail']:10.1f}%{m['goalward/best']:10.1f}%"
              f"{m['enrich/points']:+11.1f}{100*m['enrich/frac_of_exact']:11.0f}%")
    return rows


def enrichment_curves(prefix=PREFIX, order=None):
    """enrich/* over training, for runs that tracked it. Older runs simply do not appear."""
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    drew = False
    for c, name in enumerate(names):
        hists = runs[name]
        if "enrich/points" not in hists[0][-1]:
            continue
        steps = [m["step"] for m in hists[0]]
        for j, key in enumerate(["enrich/points", "goalward/best"]):
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            ax[j].plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
        ys = np.array([[m.get("goalward/fail", np.nan) for m in h] for h in hists], dtype=float)
        ax[1].plot(steps, np.nanmean(ys, 0), color=f"C{c}", ls=":", lw=1)
        drew = True
    if not drew:
        print("no run tracked enrich/* yet -- retrain, or use enrichment_table() on the checkpoints")
    ax[0].axhline(0, color="k", lw=.8, ls=":")
    ax[0].set_title("enrichment (points of goalward mass)", fontsize=9)
    ax[1].set_title("goalward mass: ask best (solid) vs ask fail (dotted)", fontsize=9)
    for a in ax:
        a.set_xlabel("step"); a.grid(alpha=.3); a.legend(fontsize=7)
    fig.tight_layout()
    return fig


if RUN_SPEED_SWEEP:
    _L = LOSSES; LOSSES = list(SPEED_SPECS)
    _ = enrichment_table(SPEED_PREFIX); enrichment_curves(SPEED_PREFIX); plt.show()
    LOSSES = _L
_ = enrichment_table()
fig = enrichment_curves(); _mark(fig)
plt.show()

## 4. Collapse diagnostics

Under counterfactual queries `info_gain` is expected to go negative (the row's evidence lowers belief in a bin it did not reach); it is no longer the flatness signal. `cond_gap` -> 0 still means R is being ignored.

In [ ]:
def plot_diagnostics(prefix, keys=(("tf", "data: teacher-forced next-token loss", True),
                                   ("cons", "consistency objective", True),
                                   ("cond_gap", "cond_gap: mean |v_t - u_t|   (-> 0 = R ignored)", False),
                                   ("info_gain", "info_gain: log q_n(R) - log q_0(R)   (-> 0 = head flat)", False)),
                     order=None, figsize=(4.6, 3.8)):
    """Train-side view. keys is (history key, panel title, log y). Configs with no consistency term simply
    do not appear in the cons/cond_gap/info_gain panels."""
    runs = load_history(prefix, "train")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(1, len(keys), figsize=(figsize[0] * len(keys), figsize[1]), squeeze=False)
    for j, (key, title, logy) in enumerate(keys):
        a = ax[0, j]
        for c, name in enumerate(names):
            hists = runs[name]
            steps = [m["step"] for m in hists[0]]
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            if np.isnan(ys).all():
                continue
            a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
            if len(hists) > 1:
                a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
        a.set_yscale("log") if logy else a.axhline(0, color="k", lw=.8, ls=":")
        a.set_title(title, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    return fig


fig = plot_diagnostics(PREFIX); _mark(fig)
plt.show()

## 5. What the late samplers actually proposed

Per-call readouts recorded by the samplers during training: query-bin histogram, how many rows the belief floor cut and how many steps survived, and for rollouts how often the model ended its own trajectory.

In [ ]:
def sampler_readout(run):
    s = SAMPLERS.get(run)
    if s is None:
        print(run, "was skipped (already on disk); readouts exist only for runs trained in this session"); return
    for sub in s.late:
        h = sub.history
        if not h:
            print("  no calls yet"); continue
        bins = np.stack([x["bins"] for x in h])
        print(f"  {run}: {len(h)} calls; query-bin share (mean):", np.round(bins.sum(0) / bins.sum(), 3))
        print("    mean used steps %.1f   rows cut by the floor %.2f" % (
            np.mean([x["mean_used"] for x in h]), np.mean([x["n_cut"] for x in h]) / bins.sum(1).mean()))
        if "ended" in h[0]:
            print("    rollouts ended by END %.2f   mean rollout length %.1f" % (
                np.mean([x["ended"] for x in h]), np.mean([x["mean_len"] for x in h])))

for name in LOSSES:
    sampler_readout(f"{PREFIX}/{name}_s0")


## 6. Far-start value tail

Evaluation only (uses the exact DP). The number the identity is supposed to repair: the reward head's mass on the top bins at far starts, which should fall by orders of magnitude with distance. `KL` is the mean far-start KL(exact || model) at the start prefix.

In [ ]:
GT = compute_ground_truth(MAZE)
cells = np.flatnonzero((MAZE.dist >= 0) & (MAZE.dist < 10**6)); dist = MAZE.dist[cells]
pos0 = np.zeros((len(cells), MAZE.T + 1), np.int64); pos0[:, 0] = cells
X2 = TOK.with_mode(TOK.encode_body(pos0, np.zeros((len(cells), MAZE.T), np.int64), np.zeros(len(cells), np.int64)), None)[:, :2]
H0 = GT.h[0][cells]

def q0_table(prefix=PREFIX, names=None, seed=0, bins=None, dists=(5, 8, 10, 12, 15, 18, 20)):
    bins = bins or tuple(FAR_BINS[::-1])                          # far starts' own best bins
    Q = {}
    for name in names or LOSSES:
        try:
            params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
        except FileNotFoundError:
            continue
        v = np.asarray(MazeTransformer(mcfg).apply({"params": params}, jnp.asarray(X2), jnp.asarray(TOK.types[:2]))["value"][:, 1]).astype(np.float64)
        q = np.exp(v - v.max(-1, keepdims=True)); Q[name] = q / q.sum(-1, keepdims=True)
    far = dist >= 10
    for b in bins:
        print(f"\nq0(bin {b}) by start distance" + " " * 12 + "".join(f"{n:>14}" for n in Q) + f"{'exact':>14}")
        for dd in dists:
            m = dist == dd
            print(f"  dist {dd:2d}" + " " * 22 + "".join(f"{Q[n][m, b].mean():14.2e}" for n in Q) + f"{H0[m, b].mean():14.2e}")
    print("\nfar-start KL(exact || model): " + "  ".join(f"{n} {(H0[far] * (np.log(H0[far] + 1e-30) - np.log(Q[n][far] + 1e-30))).sum(-1).mean():.4f}" for n in Q))

q0_table()
